# Lab Day 2 — DeepWeeds: backbone, huấn luyện và suy luận

Notebook này làm theo GUIDE: kiểm tra split và EDA, kiểm tra pipeline, so sánh backbone, ablation có kiểm soát, so sánh inference trên validation, khóa cấu hình rồi đánh giá test một lần cho mỗi seed.

Chạy trên Colab/Kaggle có GPU và Internet. Chỉ dùng validation để chọn cấu hình. Không dùng số tham khảo trong bài báo làm kết quả thực nghiệm.


## 0. Cài đặt và kết nối repo

In [ ]:
%pip install -q timm fvcore openpyxl

In [ ]:
import base64
import io
import zipfile
from pathlib import Path

SOURCE_BUNDLE_B64 = (
    '...'
    'bnRzLnR4dFBLBQYAAAAACQAJAPgCAAAPkwAAAAA='
)

bundle_data = base64.b64decode(SOURCE_BUNDLE_B64)
with zipfile.ZipFile(io.BytesIO(bundle_data)) as source_bundle:
    members = source_bundle.namelist()
    if any(name.startswith(('/', '\\\\')) or '..' in Path(name).parts for name in members):
        raise ValueError('Unsafe path in embedded source bundle')
    source_bundle.extractall(Path.cwd())
print(f'Installed {len(members)} project files into {Path.cwd()}')
del SOURCE_BUNDLE_B64, bundle_data, source_bundle, members


In [ ]:
from pathlib import Path
import hashlib
import json
import os
import platform
import random
import shutil
import subprocess
import sys
import urllib.request
import zipfile

import numpy as np
import pandas as pd
import torch

REPO_DIR = Path.cwd().resolve()
if not (REPO_DIR / "eval.py").is_file():
    candidates = [
        Path("/content/deepweeds-lab"),
        Path("/kaggle/working/deepweeds-lab"),
    ]
    REPO_DIR = next((path for path in candidates if (path / "eval.py").is_file()), None)
if REPO_DIR is None:
    raise FileNotFoundError("Đặt REPO_DIR trỏ tới thư mục repo có eval.py rồi chạy lại ô này.")

SUBMISSION_DIR = REPO_DIR / "submissions" / "deepweeds_lab"
CODE_DIR = SUBMISSION_DIR / "code"
DATA_DIR = REPO_DIR / "data"
LABELS_DIR = DATA_DIR / "labels"
PREDICTIONS_DIR = SUBMISSION_DIR / "predictions"
CURVES_DIR = SUBMISSION_DIR / "curves"
for path in (SUBMISSION_DIR, CODE_DIR, LABELS_DIR, PREDICTIONS_DIR, CURVES_DIR):
    path.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(REPO_DIR))
sys.path.insert(0, str(CODE_DIR))
import eval as ev
import dataset
import inference
import losses
import model as model_code
import train
import benchmark

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Python:", platform.python_version(), "| torch:", torch.__version__)
print("Device:", torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else DEVICE)
if DEVICE.type != "cuda":
    raise RuntimeError("Bật GPU trong Colab/Kaggle trước khi chạy các thí nghiệm.")

## 1. Tải dữ liệu, xác minh và EDA

In [ ]:
DATA_DIR.mkdir(parents=True, exist_ok=True)
LABELS_DIR.mkdir(parents=True, exist_ok=True)
zip_path = DATA_DIR / "images.zip"
local_archive = REPO_DIR / "archive.zip"
kaggle_input = Path("/kaggle/input/images")
kaggle_archives = sorted(kaggle_input.rglob("*.zip")) if kaggle_input.is_dir() else []
kaggle_images = sorted(
    path for path in kaggle_input.rglob("*")
    if path.is_file() and path.suffix.lower() == ".jpg"
) if kaggle_input.is_dir() else []

archive_source = None
archive_is_official = False
if kaggle_archives:
    archive_source = next((path for path in kaggle_archives if path.name.lower() == "images.zip"), kaggle_archives[0])
elif local_archive.is_file():
    archive_source = local_archive
elif len(kaggle_images) < 17_509 and not zip_path.is_file():
    image_url = "https://zenodo.org/records/7939060/files/images.zip?download=1"
    urllib.request.urlretrieve(image_url, zip_path)
    archive_source = zip_path
    archive_is_official = True
elif zip_path.is_file():
    archive_source = zip_path
    archive_is_official = True

actual_md5 = None
if archive_source is not None:
    digest = hashlib.md5()
    with archive_source.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1 << 20), b""):
            digest.update(chunk)
    actual_md5 = digest.hexdigest()
    expected_md5 = "b7b30f96d466fba86016aa5a26606e0f"
    if archive_is_official and actual_md5 != expected_md5:
        raise ValueError(f"images.zip MD5 mismatch: {actual_md5}")

    with zipfile.ZipFile(archive_source) as archive:
        archive.extractall(DATA_DIR)

label_base = "https://raw.githubusercontent.com/AlexOlsen/DeepWeeds/master/labels"
for name in ("labels", "train_subset0", "val_subset0", "test_subset0"):
    target = LABELS_DIR / f"{name}.csv"
    if not target.is_file():
        urllib.request.urlretrieve(f"{label_base}/{name}.csv", target)

image_files = list(DATA_DIR.rglob("*.jpg")) + list(DATA_DIR.rglob("*.JPG"))
if kaggle_images and archive_source is None:
    parent_counts = {}
    for image_path in kaggle_images:
        parent_counts[image_path.parent] = parent_counts.get(image_path.parent, 0) + 1
    source_root, root_count = max(parent_counts.items(), key=lambda item: item[1])
    if root_count >= 17_509:
        IMAGES_DIR = source_root
        image_files = kaggle_images
    else:
        IMAGES_DIR = DATA_DIR / "images"
        IMAGES_DIR.mkdir(parents=True, exist_ok=True)
        seen_names = set()
        for source in kaggle_images:
            if source.name in seen_names:
                raise ValueError(f"Duplicate image basename in Kaggle input: {source.name}")
            seen_names.add(source.name)
            target = IMAGES_DIR / source.name
            if not target.exists():
                try:
                    target.symlink_to(source)
                except OSError:
                    shutil.copy2(source, target)
        image_files = list(IMAGES_DIR.glob("*.jpg")) + list(IMAGES_DIR.glob("*.JPG"))
else:
    IMAGES_DIR = DATA_DIR / "images" if (DATA_DIR / "images").is_dir() else DATA_DIR
if len(image_files) < 17_509:
    raise FileNotFoundError(f"Found only {len(image_files)} jpg files after data setup.")

train_df, val_df, test_df = dataset.load_split(LABELS_DIR, fold=0)
split_report = dataset.check_split(train_df, val_df, test_df, IMAGES_DIR)
(SUBMISSION_DIR / "split_validation.json").write_text(
    json.dumps(split_report, indent=2, ensure_ascii=False), encoding="utf-8"
)
print("Image source:", archive_source if archive_source is not None else "direct Kaggle input")
print("Archive MD5:", actual_md5 if actual_md5 else "not applicable")
print("Images found:", len(image_files))

In [ ]:
import matplotlib.pyplot as plt
from dataset import CLASS_NAMES, DeepWeedsDataset

counts = pd.DataFrame({
    split: frame["Label"].astype(int).value_counts().reindex(range(9), fill_value=0)
    for split, frame in (("train", train_df), ("val", val_df), ("test", test_df))
})
counts.index = CLASS_NAMES
display(counts)
print("Largest/smallest class ratio:", round(counts["train"].max() / counts["train"].min(), 3))

ax = counts.plot(kind="bar", figsize=(13, 5), rot=35)
ax.set_ylabel("Image count")
ax.set_title("DeepWeeds fold 0 class distribution")
plt.tight_layout()
plt.savefig(SUBMISSION_DIR / "class_distribution.png", dpi=160)
plt.show()

sample_set = DeepWeedsDataset(train_df, IMAGES_DIR)
rng = np.random.default_rng(2026)
fig, axes = plt.subplots(9, 3, figsize=(10, 24))
for label, class_name in enumerate(CLASS_NAMES):
    candidates = np.flatnonzero(train_df["Label"].astype(int).to_numpy() == label)
    chosen = rng.choice(candidates, size=min(3, len(candidates)), replace=False)
    for column, index in enumerate(chosen):
        image = sample_set._paths[int(index)]
        axes[label, column].imshow(plt.imread(image))
        axes[label, column].axis("off")
        axes[label, column].set_title(class_name if column == 0 else "")
plt.suptitle("Three train examples per class (fold 0)")
plt.tight_layout()
plt.savefig(SUBMISSION_DIR / "eda_samples.png", dpi=160)
plt.show()

## 2. Kiểm tra pipeline trước khi chạy sweep

In [ ]:
train.set_seed(2026)
smoke_loader = dataset.make_loader(
    train_df, IMAGES_DIR, dataset.build_transforms(True, 224, "basic"),
    batch_size=8, train=True, num_workers=2,
)
images, labels, filenames = next(iter(smoke_loader))
probe = model_code.build_model("resnet50", pretrained=True, init="finetune").to(DEVICE).eval()
with torch.inference_mode():
    initial_logits = probe(images.to(DEVICE))
    initial_loss = torch.nn.functional.cross_entropy(initial_logits, labels.to(DEVICE)).item()
print("Initial CE, expected near ln(9)=2.197:", round(initial_loss, 4))
print("Batch:", tuple(images.shape), "| labels:", tuple(labels.shape), "| first file:", filenames[0])

mean = torch.tensor(dataset.IMAGENET_MEAN).view(1, 3, 1, 1)
std = torch.tensor(dataset.IMAGENET_STD).view(1, 3, 1, 1)
display_images = (images[:8] * std + mean).clamp(0, 1)
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for axis, image, label in zip(axes.flat, display_images, labels[:8]):
    axis.imshow(image.permute(1, 2, 0).cpu().numpy())
    axis.set_title(CLASS_NAMES[int(label)])
    axis.axis("off")
fig.tight_layout()
fig.savefig(SUBMISSION_DIR / "augmented_batch.png", dpi=160)
plt.show()

torch.manual_seed(0)
toy_logits = torch.randn(32, 9, device=DEVICE)
toy_labels = torch.randint(0, 9, (32,), device=DEVICE)
ce = torch.nn.functional.cross_entropy(toy_logits, toy_labels)
focal_gamma_zero = losses.FocalLoss(gamma=0.0)(toy_logits, toy_labels)
focal_error = float((ce - focal_gamma_zero).abs().item())
print("Focal gamma=0 versus CE absolute error:", focal_error)
assert focal_error < 1e-6

mixed, targets = losses.mix_batch(images.to(DEVICE), labels.to(DEVICE), alpha=1.0, mode="cutmix")
assert mixed.shape == images.to(DEVICE).shape
assert 0.0 <= targets[2] <= 1.0
print("CutMix shape:", tuple(mixed.shape), "| actual lambda:", round(targets[2], 4))

fused_probe = inference.fuse_conv_bn(model_code.build_model(
    "resnet50", pretrained=True, init="finetune"
).to(DEVICE).eval())

In [ ]:
# Overfit a fixed mini-batch to catch label/model/gradient mistakes before long runs.
small_model = model_code.build_model(
    "mobilenetv3_large_100", pretrained=False, init="scratch"
).to(DEVICE)
small_model.train()
fixed_x = images[:8].to(DEVICE)
fixed_y = labels[:8].to(DEVICE)
small_optimizer = torch.optim.AdamW(small_model.parameters(), lr=1e-3, weight_decay=0.0)
loss_trace = []
for step in range(120):
    small_optimizer.zero_grad(set_to_none=True)
    loss = torch.nn.functional.cross_entropy(small_model(fixed_x), fixed_y)
    loss.backward()
    small_optimizer.step()
    if step in (0, 19, 39, 59, 79, 99, 119):
        loss_trace.append((step + 1, float(loss.detach().item())))
print("Fixed-batch overfit trace:", loss_trace)
assert loss_trace[-1][1] < loss_trace[0][1], "Mini-batch loss did not decrease."

## 3. Bước 1 — So sánh năm backbone trên cùng công thức nền

In [ ]:
COMMON = {
    "fold": 0,
    "img_size": 224,
    "epochs": 12,
    "batch_size": 64,
    "lr_backbone": 1e-4,
    "lr_head": 1e-3,
    "weight_decay": 0.05,
    "warmup_epochs": 1.0,
    "amp": True,
    "channels_last": True,
    "num_workers": 2,
    "images_dir": str(IMAGES_DIR),
    "labels_dir": str(LABELS_DIR),
    "out_dir": str(SUBMISSION_DIR / "runs"),
    "pred_dir": str(PREDICTIONS_DIR),
    "curves_dir": str(CURVES_DIR),
}
BACKBONES = [
    ("B01", "resnet50"),
    ("B02", "resnext50_32x4d"),
    ("B03", "convnext_tiny"),
    ("B04", "deit_small_patch16_224"),
    ("B05", "mobilenetv3_large_100"),
]
manifest = {}
backbone_results = []
backbone_latencies = {}
for exp_id, backbone in BACKBONES:
    result = train.run(train.Config(exp_id=exp_id, seed=0, backbone=backbone, **COMMON))
    backbone_results.append(result)
    probe_model = model_code.build_model(backbone, pretrained=False, init="finetune").to(DEVICE).eval()
    probe_checkpoint = torch.load(
        SUBMISSION_DIR / "runs" / exp_id / "seed0" / "checkpoint_best.pt",
        map_location=DEVICE, weights_only=True,
    )
    probe_model.load_state_dict(probe_checkpoint["model_state"])
    backbone_latencies[exp_id] = benchmark.latency_report(
        probe_model, 1, 224, dtype="fp32", device=str(DEVICE), warmup=10, iters=50
    )
    del probe_model
    torch.cuda.empty_cache()
    manifest[exp_id] = {"axis": "backbone", "difference": f"baseline recipe; {backbone}"}
    print(exp_id, backbone, "val macro-F1=", round(result["macro_f1_val"], 4))
(SUBMISSION_DIR / "backbone_latency.json").write_text(
    json.dumps(backbone_latencies, indent=2, ensure_ascii=False), encoding="utf-8"
)
best_backbone_run = max(backbone_results, key=lambda row: row["macro_f1_val"])
BEST_BACKBONE = best_backbone_run["backbone"]
print("Selected for training ablations using validation only:", BEST_BACKBONE)

## 4. Bước 2 — Ablation từng trục và một tổ hợp

In [ ]:
baseline = train.run(train.Config(
    exp_id="T00", seed=0, backbone=BEST_BACKBONE, **COMMON
))
training_results = {"T00": baseline}
manifest["T00"] = {"axis": "baseline", "difference": "baseline recipe"}

ablation_specs = [
    ("T01", "A: initialization", {"init": "frozen"}),
    ("T02", "A: initialization", {"init": "scratch", "pretrained": False}),
    ("T03", "B: augmentation", {"aug": "color"}),
    ("T04", "B: augmentation", {"aug": "randaug"}),
    ("T05", "B: augmentation", {"mix": "cutmix", "mix_alpha": 1.0}),
    ("T06", "C: loss", {"loss": "ls", "label_smoothing": 0.1}),
    ("T07", "C: loss", {"loss": "focal", "focal_gamma": 2.0}),
    ("T08", "D: sample balance", {"sampler": "balanced"}),
]
for exp_id, axis, overrides in ablation_specs:
    difference = ", ".join(f"{key}={value}" for key, value in overrides.items())
    manifest[exp_id] = {"axis": axis, "difference": difference}
    result = train.run(train.Config(
        exp_id=exp_id, seed=0, backbone=BEST_BACKBONE, **COMMON, **overrides
    ))
    training_results[exp_id] = result
    print(exp_id, axis, "val macro-F1=", round(result["macro_f1_val"], 4))

def choose_best(ids):
    return max(ids, key=lambda name: training_results[name]["macro_f1_val"])

best_init_id = choose_best(["T00", "T01", "T02"])
best_aug_id = choose_best(["T00", "T03", "T04", "T05"])
best_loss_id = choose_best(["T00", "T06", "T07"])
init_cfg = training_results[best_init_id]["config"]
aug_cfg = training_results[best_aug_id]["config"]
loss_cfg = training_results[best_loss_id]["config"]
combo = {
    "init": init_cfg["init"],
    "pretrained": init_cfg["pretrained"],
    "aug": aug_cfg["aug"],
    "mix": aug_cfg["mix"],
    "mix_alpha": aug_cfg["mix_alpha"],
    "loss": loss_cfg["loss"],
    "label_smoothing": loss_cfg["label_smoothing"],
    "focal_gamma": loss_cfg["focal_gamma"],
}
manifest["T09"] = {
    "axis": "A+B+C combination",
    "difference": f"best A={best_init_id}, B={best_aug_id}, C={best_loss_id} by validation",
    "notes": "Combined validation-selected factors; confirm interactions with final seeds.",
}
training_results["T09"] = train.run(train.Config(
    exp_id="T09", seed=0, backbone=BEST_BACKBONE, **COMMON, **combo
))
manifest_path = SUBMISSION_DIR / "experiment_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
print("Combined recipe:", combo)

## 5. Bước 3 — So sánh inference chỉ trên validation

In [ ]:
best_train_id = max(training_results, key=lambda name: training_results[name]["macro_f1_val"])
best_train_summary = training_results[best_train_id]
best_cfg = best_train_summary["config"]
checkpoint_path = SUBMISSION_DIR / "runs" / best_train_id / "seed0" / "checkpoint_best.pt"
inference_model = model_code.build_model(
    best_cfg["backbone"], pretrained=False, init=best_cfg["init"],
    drop_rate=best_cfg["drop_rate"],
).to(DEVICE).eval()
checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=True)
inference_model.load_state_dict(checkpoint["model_state"])

val_loader = dataset.make_loader(
    val_df, IMAGES_DIR, dataset.build_transforms(False, 224),
    batch_size=64, train=False, num_workers=2,
)
crop_val_loader = dataset.make_loader(
    val_df, IMAGES_DIR, dataset.build_transforms(False, 256),
    batch_size=64, train=False, num_workers=2,
)
names0, y_val, logits0 = inference.predict_logits(inference_model, val_loader, DEVICE)
names_flip, y_flip, scores_flip_prob = inference.predict_multi_logits(
    inference_model, val_loader, DEVICE,
    lambda batch: [batch, inference.view_hflip(batch)], space="prob",
)
_, _, scores_flip_logit = inference.predict_multi_logits(
    inference_model, val_loader, DEVICE,
    lambda batch: [batch, inference.view_hflip(batch)], space="logit",
)
names_crop, y_crop, scores_crop = inference.predict_multi_logits(
    inference_model, crop_val_loader, DEVICE,
    lambda batch: inference.views_multicrop(batch, 224), space="prob",
)
assert names0 == names_flip == names_crop
assert np.array_equal(y_val, y_flip) and np.array_equal(y_val, y_crop)

def score_logits(scores, labels):
    probs = torch.as_tensor(scores, dtype=torch.float64).softmax(dim=1).numpy()
    return ev.compute_metrics(labels, probs.argmax(axis=1), probs), probs

m00, p00 = score_logits(logits0, y_val)
m01, p01 = score_logits(scores_flip_prob, y_val)
m02, p02 = score_logits(scores_crop, y_val)
m03p, p03p = score_logits(scores_flip_prob, y_val)
m03l, p03l = score_logits(scores_flip_logit, y_val)
temperature = inference.fit_temperature(logits0, y_val)
p07 = inference.apply_temperature(logits0, temperature)
m07 = ev.compute_metrics(y_val, p07.argmax(axis=1), p07)
print("I00", m00["macro_f1"], m00["ece"])
print("I01 horizontal flip", m01["macro_f1"], m01["ece"])
print("I02 five crops", m02["macro_f1"], m02["ece"])
print("I03 probability average", m03p["macro_f1"], m03p["ece"])
print("I03 logit average", m03l["macro_f1"], m03l["ece"])
print("I07 temperature", temperature, "ECE:", m07["ece"])

In [ ]:
lat00 = benchmark.latency_report(
    inference_model, 1, 224, dtype="fp32", device=str(DEVICE), warmup=10, iters=50
)
lat01 = benchmark.tta_latency(
    inference_model, 2, batch_size=1, img_size=224, dtype="fp32",
    device=str(DEVICE), warmup=10, iters=50
)
lat02 = benchmark.tta_latency(
    inference_model, 5, batch_size=1, img_size=224, dtype="fp32",
    device=str(DEVICE), warmup=10, iters=50
)
try:
    lat32 = __import__("benchmark").latency_report(
        inference_model, 32, 224, dtype="fp32", device=str(DEVICE), warmup=10, iters=50
    )
except RuntimeError as error:
    lat32 = None
    print("Batch-32 latency could not be measured on this GPU:", error)

def make_inference_row(exp_id, method, metrics, timing, k, notes=""):
    return {
        "exp_id": exp_id,
        "method": method,
        "model_checkpoint": str(checkpoint_path),
        "K": k,
        "macro_f1_val": float(metrics["macro_f1"]),
        "top1_val": float(metrics["top1"]),
        "ECE_val": float(metrics["ece"]),
        "latency_p50_ms": float(timing["p50"]),
        "latency_p95_ms": float(timing["p95"]),
        "latency_p99_ms": float(timing["p99"]),
        "throughput_images_s": float(timing["images_per_s"]),
        "relative_cost_vs_I00": float(timing["p50"] / lat00["p50"]),
        "GPU": timing["gpu"],
        "dtype": timing["dtype"],
        "BN_fused": timing["bn_fused"],
        "batch": timing["batch"],
        "img_size": timing["img_size"],
        "includes_preprocessing": False,
        "torch": timing["torch"],
        "notes": notes,
    }

inference_rows = [
    make_inference_row("I00", "1-view FP32", m00, lat00, 1, "Validation; preprocessing excluded."),
    make_inference_row("I01", "horizontal-flip TTA; probability mean", m01, lat01, 2),
    make_inference_row("I02", "five-crop TTA; probability mean", m02, lat02, 5),
    make_inference_row("I03", "horizontal-flip; probability mean", m03p, lat01, 2),
    make_inference_row("I03", "horizontal-flip; logit mean", m03l, lat01, 2),
    make_inference_row("I07", "temperature scaling on I00", m07, lat00, 1,
                       f"T={temperature:.6g}; fitted on validation only."),
]
if lat32 is not None:
    inference_rows.append({
        **make_inference_row("I00B32", "1-view FP32 batch 32", m00, lat32, 1),
        "notes": "Throughput batch; model quality uses the same validation predictions.",
    })
(SUBMISSION_DIR / "inference_results.json").write_text(
    json.dumps(inference_rows, indent=2, ensure_ascii=False), encoding="utf-8"
)
display(pd.DataFrame(inference_rows)[[
    "exp_id", "method", "K", "macro_f1_val", "ECE_val",
    "latency_p50_ms", "latency_p95_ms", "latency_p99_ms",
]])

## 6. Bước 4 — Khóa lựa chọn trên validation rồi chạy final/test

In [ ]:
eligible = [row for row in inference_rows if row["exp_id"] != "I00B32"]
selected_inference = max(
    eligible, key=lambda row: (row["macro_f1_val"], -row["latency_p95_ms"])
)
method = selected_inference["method"]
if method.startswith("horizontal-flip"):
    selected_tta = "hflip"
    selected_space = "logit" if "logit mean" in method else "prob"
elif method.startswith("five-crop"):
    selected_tta = "multicrop"
    selected_space = "prob"
else:
    selected_tta = "identity"
    selected_space = "prob"

print("Locked training recipe by validation:", best_train_id, best_cfg)
print("Locked inference by validation:", selected_inference["exp_id"], method)
print("Validation macro-F1:", selected_inference["macro_f1_val"])
print("Final test will run once per seed. Do not rerun or change this selection after viewing test scores.")

In [ ]:
recipe_fields = [
    "backbone", "pretrained", "init", "drop_rate", "img_size", "aug", "sampler",
    "mix", "mix_alpha", "loss", "label_smoothing", "focal_gamma",
    "class_weight_beta", "optimizer", "epochs", "batch_size", "lr_backbone",
    "lr_head", "weight_decay", "warmup_epochs", "ema_decay", "amp",
    "channels_last", "grad_clip_norm", "num_workers",
]
final_recipe = {key: best_cfg[key] for key in recipe_fields}
final_recipe.update({
    "tta": selected_tta,
    "tta_space": selected_space,
    "temperature_scaling": True,
    "save_test_predictions": True,
    "save_uncalibrated_predictions": True,
})

final_common = {key: value for key, value in COMMON.items() if key not in set(recipe_fields)}
for seed in (0, 1, 2):
    final_summary = train.run(train.Config(
        exp_id="F01", seed=seed, **final_common, **final_recipe
    ))
    print("F01 validation seed", seed, final_summary["macro_f1_val"])

for seed in (0, 1, 2):
    baseline_summary = train.run(train.Config(
        exp_id="T00", seed=seed, backbone=BEST_BACKBONE, **COMMON,
        save_test_predictions=True, save_uncalibrated_predictions=False,
        temperature_scaling=False, tta="identity",
    ))
    print("T00 validation seed", seed, baseline_summary["macro_f1_val"])

print("All final and baseline test predictions have been saved. Do not rerun this cell.")

In [ ]:
# Post-final error analysis: inspect predictions only after the selection is locked.
from dataset import CLASS_NAMES, DeepWeedsDataset
import matplotlib.pyplot as plt

final_seed0 = ev.read_pred(str(PREDICTIONS_DIR / "F01_seed0_test.csv"))
confusion = ev.confusion_matrix(final_seed0.y_true, final_seed0.y_pred)
confusion_frame = pd.DataFrame(confusion, index=CLASS_NAMES, columns=CLASS_NAMES)
display(confusion_frame)
fig, ax = plt.subplots(figsize=(10, 9))
image = ax.imshow(confusion, cmap="Blues")
ax.set_xticks(range(9), CLASS_NAMES, rotation=55, ha="right")
ax.set_yticks(range(9), CLASS_NAMES)
ax.set_xlabel("Predicted label")
ax.set_ylabel("True label")
ax.set_title("F01 seed 0 test confusion matrix (counts)")
fig.colorbar(image, ax=ax)
fig.tight_layout()
fig.savefig(SUBMISSION_DIR / "confusion_matrix_F01.png", dpi=160)
plt.show()

error_rows = np.flatnonzero(final_seed0.y_true != final_seed0.y_pred)
print("Misclassified test images in seed 0:", len(error_rows))
error_frame = pd.DataFrame({
    "Filename": final_seed0.filenames[error_rows[:12]],
    "Label": final_seed0.y_true[error_rows[:12]],
})
error_images = DeepWeedsDataset(error_frame, IMAGES_DIR)
fig, axes = plt.subplots(3, 4, figsize=(14, 10))
for axis, row_index, image_index in zip(axes.flat, error_rows[:12], range(min(12, len(error_rows)))):
    image, _, filename = error_images[image_index]
    true_name = CLASS_NAMES[int(final_seed0.y_true[row_index])]
    predicted_name = CLASS_NAMES[int(final_seed0.y_pred[row_index])]
    axis.imshow(image)
    axis.set_title(f"true: {true_name}\npred: {predicted_name}", fontsize=8)
    axis.axis("off")
fig.tight_layout()
fig.savefig(SUBMISSION_DIR / "error_examples_F01.png", dpi=160)
plt.show()

## 7. Chấm bằng eval.py và tạo sản phẩm

In [ ]:
def run_eval(args):
    subprocess.run([sys.executable, str(REPO_DIR / "eval.py"), *args], cwd=REPO_DIR, check=True)

test_csv = str(LABELS_DIR / "test_subset0.csv")
labels_csv = str(LABELS_DIR / "labels.csv")
final_pattern = str(PREDICTIONS_DIR / "F01_seed*_test.csv")
baseline_pattern = str(PREDICTIONS_DIR / "T00_seed*_test.csv")
run_eval([
    "score", "--pred", final_pattern, "--test-csv", test_csv,
    "--labels", labels_csv, "--tag", "F01", "--out", str(SUBMISSION_DIR / "eval_out"),
])
run_eval([
    "score", "--pred", baseline_pattern, "--test-csv", test_csv,
    "--labels", labels_csv, "--tag", "T00", "--out", str(SUBMISSION_DIR / "eval_out"),
])
run_eval([
    "grade", "--final", final_pattern, "--baseline", baseline_pattern,
    "--uncal", str(PREDICTIONS_DIR / "F01_seed*_test_uncal.csv"),
    "--final-val", str(PREDICTIONS_DIR / "F01_seed*_val.csv"),
    "--test-csv", test_csv, "--val-csv", str(LABELS_DIR / "val_subset0.csv"),
    "--labels", labels_csv, "--out", str(SUBMISSION_DIR / "eval_out"),
])
subprocess.run(
    [sys.executable, str(CODE_DIR / "build_results.py"), "--report"],
    cwd=REPO_DIR, check=True,
)
print("Generated:", SUBMISSION_DIR / "results.xlsx", "and", SUBMISSION_DIR / "report.md")

## Checklist trước khi nộp

- Xác minh fold 0: đủ 17.509 tên file duy nhất, các giao rỗng và mọi ảnh tồn tại.
- Có ít nhất năm backbone: ResNet, ResNeXt, ConvNeXt, DeiT và MobileNet.
- So sánh các trục khởi tạo, augmentation và loss; ghi nhận sampler và tổ hợp.
- Đo ít nhất bốn phương pháp inference cùng latency p50/p95/p99.
- Chạy T00 baseline và F01 final đủ ba seed; lưu prediction đúng định dạng.
- Chạy eval.py score/grade; tạo workbook bảy sheet, curves và báo cáo.
- Đối chiếu mọi kết luận với log thật; không chọn cấu hình dựa trên test.
